In [13]:
import os
import cv2
import numpy as np
import librosa
from imutils import face_utils
import dlib
import pandas as pd

In [14]:
base_audio = "C:/Users/connellj2/OneDrive - Wentworth Institute of Technology/Documents/DecepTech/CroppedAudio/baseline.wav"
base_vid = "C:/Users/connellj2/OneDrive - Wentworth Institute of Technology/Documents/DecepTech/CroppedClips/baseline.avi"

In [15]:
def extract_mfcc(audio_path):
    y, sr = librosa.load(audio_path, sr=None)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    return np.mean(mfcc, axis=1)

baseline_mfcc = extract_mfcc(base_audio)

baseline_audio_df = pd.DataFrame([{
    **{f"mfcc_{i}": baseline_mfcc[i] for i in range(13)}
}])

In [16]:
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")

frame_interval = 15
cap = cv2.VideoCapture(base_vid)
frame_count = 0
sampled_landmarks = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    if frame_count % frame_interval != 0:
        continue

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY).astype("uint8")
    faces = detector(gray)

    if len(faces) == 0:
        continue

    for rect in faces:
        shape = predictor(gray, rect)
        shape_np = face_utils.shape_to_np(shape)
        sampled_landmarks.append(shape_np)

cap.release()

# Motion tracking
motion_sequence = [
    np.linalg.norm(sampled_landmarks[i+1] - sampled_landmarks[i], axis=1).sum()
    for i in range(len(sampled_landmarks) - 1)
]

smoothed_motion = np.convolve(motion_sequence, np.ones(3)/3, mode="valid") if len(motion_sequence) >= 3 else motion_sequence
smoothed_avg_motion = np.mean(smoothed_motion) if len(smoothed_motion) > 0 else 0


In [18]:
# 🎯 Audio prediction
baseline_audio_df["audio_pred"] = audio_model.predict_proba(baseline_audio_df)[:, 1]
baseline_audio_score = baseline_audio_df["audio_pred"].iloc[0]

# 🧠 Facial motion score (already computed)
baseline_motion_score = smoothed_avg_motion

# 📊 Print summary
print("🔍 Baseline Deception Profile")
print(f"Audio deception score: {baseline_audio_score:.4f}")
print(f"Facial motion intensity: {baseline_motion_score:.4f}")


NameError: name 'audio_model' is not defined